# 21 — Full Backtest

Run the backtest engine across ALL completed GAME markets. Aggregate Brier scores, PnL, and edge stats.

In [ ]:
import sys
sys.path.insert(0, '/Users/chriswang/Desktop/prediction-market-exploration/nba-edge')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from nba_edge.data.s3_reader import discover_trade_dates, load_trades, load_boxscores_for_game, discover_game_ids
from nba_edge.data.ticker_parser import parse_ticker, is_game_winner
from nba_edge.data.game_alignment import match_ticker_to_game
from nba_edge.backtest.engine import BacktestEngine
from nba_edge.backtest.metrics import simulate_pnl, compute_brier_scores
from nba_edge.models.analytical import AnalyticalWinProb

In [ ]:
# Load all trades
dates = discover_trade_dates(min_date='2026-04-21')
df = load_trades(dates)
print(f'Total trades: {len(df):,}')

# Find all GAME tickers
game_tickers = [t for t in df['market_ticker'].unique().to_list() if is_game_winner(t)]
print(f'GAME tickers: {len(game_tickers)}')

In [ ]:
# Run backtest for each GAME ticker
engine = BacktestEngine(model=AnalyticalWinProb(variance_rate=0.44))
results = []
skipped = []

for ticker in sorted(game_tickers):
    parsed = parse_ticker(ticker)
    if not parsed.game_date:
        skipped.append((ticker, 'no game_date parsed'))
        continue
    
    # Find game
    try:
        available = discover_game_ids(parsed.game_date)
    except Exception as e:
        skipped.append((ticker, f'discover failed: {e}'))
        continue
    
    matched = match_ticker_to_game(ticker, available)
    if not matched:
        skipped.append((ticker, 'no game match'))
        continue
    
    # Load boxscores
    snapshots = load_boxscores_for_game(matched, parsed.game_date)
    if len(snapshots) < 50:
        skipped.append((ticker, f'only {len(snapshots)} snapshots'))
        continue
    
    # Check game is complete
    if snapshots[-1].get('game_status', 0) != 3:
        skipped.append((ticker, 'game not complete'))
        continue
    
    # Run backtest
    ticker_trades = df.filter(pl.col('market_ticker') == ticker)
    result = engine.run_game(ticker_trades, snapshots, ticker)
    if result and len(result.trades) > 0:
        result.game_id = matched
        results.append(result)
        print(f'  {ticker:45s} | {len(result.trades):>5} trades | outcome={result.outcome_yes}')
    else:
        skipped.append((ticker, 'alignment failed'))

print(f'\nBacktested: {len(results)} markets')
print(f'Skipped: {len(skipped)}')
if skipped:
    for t, reason in skipped[:10]:
        print(f'  {t}: {reason}')

In [ ]:
# Aggregate Brier scores
brier = compute_brier_scores(results)
print('=== Brier Scores (lower = better) ===')
print(f"  Model:  {brier['model_brier']:.4f}")
print(f"  Market: {brier['market_brier']:.4f}")
print(f"  Observations: {brier['n_observations']:,}")
print()
if brier['model_brier'] < brier['market_brier']:
    print('  → Model is MORE accurate than the market!')
else:
    print('  → Market is more accurate than our model.')

In [ ]:
# PnL at various thresholds
thresholds = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20]
pnl_results = []

print(f"{'Threshold':>10s} {'Trades':>8s} {'PnL ($)':>10s} {'Win Rate':>10s} {'Avg Edge':>10s}")
print('-' * 55)

for thresh in thresholds:
    pnl = simulate_pnl(results, edge_threshold=thresh)
    pnl_results.append(pnl)
    print(f'{thresh:>10.2f} {pnl.n_trades_taken:>8d} {pnl.gross_pnl_cents/100:>10.2f} '
          f'{pnl.win_rate:>10.1%} {pnl.avg_edge_taken:>10.3f}')

In [ ]:
# Plot PnL vs threshold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pnl_dollars = [p.gross_pnl_cents / 100 for p in pnl_results]
trades_taken = [p.n_trades_taken for p in pnl_results]

axes[0].bar([f'{t:.0%}' for t in thresholds], pnl_dollars, color=['green' if p > 0 else 'red' for p in pnl_dollars])
axes[0].set_xlabel('Edge Threshold')
axes[0].set_ylabel('PnL ($)')
axes[0].set_title('Gross PnL by Edge Threshold')
axes[0].axhline(0, color='black', linestyle=':')

axes[1].bar([f'{t:.0%}' for t in thresholds], trades_taken, color='steelblue')
axes[1].set_xlabel('Edge Threshold')
axes[1].set_ylabel('Trades Taken')
axes[1].set_title('Number of Trades by Threshold')

plt.tight_layout()
plt.show()

In [ ]:
# Per-game breakdown
print(f"{'Ticker':45s} {'Trades':>7s} {'Outcome':>8s} {'Avg Edge':>10s}")
print('-' * 75)
for r in results:
    edges = [t.edge for t in r.trades]
    print(f"{r.market_ticker:45s} {len(r.trades):>7d} {'YES' if r.outcome_yes else 'NO':>8s} {np.mean(edges):>10.4f}")

In [ ]:
# Edge distribution across all games
all_edges = [t.edge * 100 for r in results for t in r.trades]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(all_edges, bins=80, range=(-30, 30), edgecolor='none', alpha=0.7)
ax.axvline(0, color='black', linestyle='-')
ax.axvline(5, color='red', linestyle='--', alpha=0.5, label='±5c')
ax.axvline(-5, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Edge (cents)')
ax.set_ylabel('Count')
ax.set_title(f'Edge Distribution — All Games ({len(results)} markets, {len(all_edges):,} trades)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean edge: {np.mean(all_edges):.2f}c')
print(f'Median edge: {np.median(all_edges):.2f}c')
print(f'Std: {np.std(all_edges):.2f}c')